# Generación de imágenes con gpt-image-2

**Lección 3 · Clase 5.2** — generar imágenes desde texto con la herramienta `image_generation` de OpenAI (Responses API), orquestada por GPT-5 a través de LangChain.

> **IMPORTANTE:** para generar imágenes OpenAI exige una **organización verificada** (subir identificación en la consola), lo que puede tomar tiempo. Además cada imagen cuesta dinero (con gpt-image-2 una 1024×1024 va de ~US$0.006 en calidad baja a ~US$0.21 en alta) — revisa [precios](https://platform.openai.com/docs/pricing).
>
> Nota de ciclo de vida: esta lección usaba `gpt-image-1` (2025); OpenAI lo deprecó en junio 2026 a favor de `gpt-image-2`. Los modelos son productos con fecha de vencimiento — por eso el nombre va en **un** parámetro.

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q langchain-openai==1.3.5 langchain-core==1.4.9 python-dotenv==1.2.2 pillow==12.3.0
from dotenv import load_dotenv
import os

# Carga OPENAI_API_KEY desde .env si existe (local); en Colab usa Secrets.
load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
except Exception:
    pass

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY presente:", True)
else:
    print("⚠️ Falta OPENAI_API_KEY — usa un archivo .env o los Secrets de Colab.")


## Cómo funciona

No llamamos a `gpt-image-2` directo: se lo damos a GPT-5 como **herramienta** (`image_generation`) vía la Responses API. El LLM interpreta el pedido, redacta el prompt final para el generador, y la imagen vuelve **en base64** dentro de la respuesta — el mismo mecanismo de tool-calling que ya conoces, pero donde el resultado del tool es una imagen.

In [ ]:
import base64
from io import BytesIO
from pathlib import Path

from IPython.display import display
from PIL import Image
from langchain_openai import ChatOpenAI

image_llm = ChatOpenAI(model="gpt-5", use_responses_api=True)
image_tool = {
    "type": "image_generation",
    "model": "gpt-image-2",
    "size": "1024x1024",
    "quality": "high",
    "background": "auto",
    "output_format": "jpeg",
    "output_compression": 80,
    "moderation": "low",
}

image_prompt = "some dogs playing poker in a photorealistic shot using a Leica camera"

image_chain = image_llm.bind_tools(
    [image_tool],
    tool_choice={"type": "image_generation"},
)

ai_msg = image_chain.invoke(f"Generate an image of {image_prompt}")

# La imagen viene en base64 dentro de la respuesta; según la versión puede llegar
# como bloque de contenido o dentro de additional_kwargs["tool_outputs"].
image_base64 = None
if isinstance(ai_msg.content, list):
    for block in ai_msg.content:
        if isinstance(block, dict) and block.get("type") == "image_generation_call":
            image_base64 = block.get("result")
            break

if not image_base64:
    for tool_output in ai_msg.additional_kwargs.get("tool_outputs", []):
        if isinstance(tool_output, dict) and tool_output.get("type") == "image_generation_call":
            image_base64 = tool_output.get("result")
            break

if image_base64:
    img = Image.open(BytesIO(base64.b64decode(image_base64)))
    display(img)
    out_dir = Path("../outputs")
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / "perros_poker.jpg"
    img.save(out_path)
    print("Imagen guardada en", out_path)
else:
    print("No image generated.")

## Ejercicio

Cambia `image_prompt` y vuelve a correr la celda. Ideas para explorar:

- **Estilo**: "óleo impresionista", "render 3D estilo Pixar", "fotografía analógica con grano".
- **Cámara/luz**: "50mm f/1.4, golden hour", "luz de neón, estilo cyberpunk".
- **Costo/velocidad**: baja `quality` a `"medium"` o `"low"` para iterar más barato, y sube a `"high"` solo para la versión final.

Fíjate en que GPT-5 **reescribe** tu pedido antes de pasárselo al generador — esa capa de interpretación es la diferencia entre darle el prompt crudo a `gpt-image-1` y orquestarlo con un LLM.